In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint06b_Multi_Period_Comparison

Purpose:
Run every strategy across three different backtest start dates -
2020, 2015, 2010 - and compare CAGR/Sharpe/drawdown side by side.
Longer history covers more market regimes (2008 crisis, 2020 COVID
crash, different rate environments), which is genuinely valuable -
but also flags a real limitation worth understanding, not just a
code detail: see the SURVIVORSHIP BIAS note below before trusting
the 2010 numbers too much.

Author:
Alison

Version:
0.12
"""

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from alpha.config import Config, DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.portfolio import calculate_monthly_returns
from alpha.regime import calculate_regime
from alpha.backtest import run_backtest
from alpha.analytics import summarize_performance

from alpha.strategies.momentum import calculate_monthly_momentum, select_top_stocks, select_bottom_stocks
from alpha.strategies.trend_following import select_trend_positions, select_downtrend_positions
from alpha.strategies.mean_reversion import select_oversold_stocks, select_overbought_stocks
from alpha.strategies.breakout import select_breakout_stocks, select_breakdown_stocks

In [ ]:
# =====================================================
# SURVIVORSHIP BIAS - READ BEFORE TRUSTING THE 2010 RESULTS
# =====================================================
# config.universe is TODAY's stock list. Testing back to 2010 asks
# "would today's AAPL/MSFT/etc. have performed well since 2010" - it
# quietly EXCLUDES any company that existed in 2010 but has since gone
# bankrupt, been delisted, or was acquired. That inflates results,
# sometimes substantially - you're only looking at the stocks that
# turned out to survive and do well enough to still be in a "top 10
# companies today" list. This isn't something the code can fix; it's
# inherent to picking today's tickers and testing them in the past.
#
# The longer-period results below are still useful for seeing how a
# STRATEGY (the momentum/trend/etc. logic itself) behaves across
# different market regimes - just don't read the absolute CAGR/growth
# numbers from the 2010 run as "what I'd have actually made", since
# real 2010-you wouldn't have known in advance which stocks would
# still be around and thriving in 2026.

In [ ]:
# =====================================================
# THREE BACKTEST PERIODS
# =====================================================

PERIODS = {
    "2020-present": Config(start_date="2020-01-01"),
    "2015-present": Config(start_date="2015-01-01"),
    "2010-present": Config(start_date="2010-01-01"),
}

In [ ]:
# =====================================================
# RUN EVERY STRATEGY, FOR EVERY PERIOD
# =====================================================
# Downloads data separately per period since each uses a different
# start_date - this cell will take a few minutes total.

all_results = {}   # {period_name: {strategy_name: BacktestResult}}
all_summaries = {} # {period_name: {strategy_name: PerformanceSummary}}
data_coverage = {} # {period_name: {ticker: first_available_date}}

for period_name, config in PERIODS.items():
    print(f"--- {period_name} (start_date={config.start_date}) ---")
    print("Downloading market data...")

    prices = get_prices(config)
    monthly_prices = get_monthly_prices(prices)
    monthly_returns = calculate_monthly_returns(monthly_prices)

    benchmark_prices = get_benchmark_prices(config)
    monthly_benchmark = benchmark_prices.resample("ME").last()
    regime = calculate_regime(monthly_benchmark, config)

    # Data coverage check - did every ticker actually have data going
    # back to the requested start_date, or did some start later (IPO,
    # data availability)? yfinance won't error if not - it just returns
    # whatever exists, starting wherever the data starts.
    data_coverage[period_name] = prices.apply(lambda col: col.first_valid_index())

    monthly_momentum = calculate_monthly_momentum(monthly_prices, config)

    signal_pairs = {
        "Momentum": (
            select_top_stocks(monthly_momentum, config),
            select_bottom_stocks(monthly_momentum, config),
        ),
        "Trend Following": (
            select_trend_positions(monthly_prices, config),
            select_downtrend_positions(monthly_prices, config),
        ),
        "Mean Reversion": (
            select_oversold_stocks(monthly_prices, config),
            select_overbought_stocks(monthly_prices, config),
        ),
        "Breakout": (
            select_breakout_stocks(monthly_prices, config),
            select_breakdown_stocks(monthly_prices, config),
        ),
    }

    period_results = {}
    period_summaries = {}

    for name, (long_signal, short_signal) in signal_pairs.items():
        result = run_backtest(monthly_returns, long_signal, short_signal, regime, config)
        period_results[name] = result
        period_summaries[name] = summarize_performance(result, monthly_prices, config)

    all_results[period_name] = period_results
    all_summaries[period_name] = period_summaries

print("\nAll periods complete.")

In [ ]:
# =====================================================
# DATA COVERAGE CHECK
# =====================================================
# For each period, when does each ticker's data actually start? If any
# date here is noticeably LATER than the period's requested start_date,
# that ticker didn't have data that far back (e.g. IPO'd after 2010) -
# it was effectively excluded from the earlier part of that backtest
# without any error being raised.

for period_name in PERIODS:
    print(f"\n{period_name}:")
    display(data_coverage[period_name])

In [ ]:
# =====================================================
# COMPARISON TABLES, ONE PER PERIOD
# =====================================================

for period_name, summaries in all_summaries.items():
    print(f"\n{'=' * 60}")
    print(period_name)
    print('=' * 60)

    summary_table = pd.DataFrame({
        name: {
            "CAGR": s.cagr,
            "Max Drawdown": s.max_drawdown,
            "Sharpe": s.sharpe_ratio,
            "Sortino": s.sortino_ratio,
            "Profit Factor": s.profit_factor,
            "Win Rate": s.win_rate,
            "Num Trades": s.num_trades,
        }
        for name, s in summaries.items()
    }).T

    display(summary_table.style.format({
        "CAGR": "{:.1%}",
        "Max Drawdown": "{:.1%}",
        "Sharpe": "{:.2f}",
        "Sortino": "{:.2f}",
        "Profit Factor": "{:.2f}",
        "Win Rate": "{:.1%}",
        "Num Trades": "{:.0f}",
    }))

In [ ]:
# =====================================================
# CAGR ACROSS PERIODS, SIDE BY SIDE
# =====================================================
# The number most worth watching here: does a strategy's CAGR stay
# roughly consistent across periods, or does it fall apart once you
# add years the strategy wasn't "tuned" against? A strategy whose CAGR
# collapses or reverses sign as you extend the window is telling you
# something important about how reliable its edge actually is.

cagr_by_period = pd.DataFrame({
    period_name: {name: s.cagr for name, s in summaries.items()}
    for period_name, summaries in all_summaries.items()
})

display(cagr_by_period.style.format("{:.1%}"))

cagr_by_period.T.plot(kind="bar", figsize=(10, 6))
plt.title("CAGR by Strategy Across Backtest Periods")
plt.ylabel("CAGR")
plt.xticks(rotation=0)
plt.grid(True, axis="y")
plt.legend(title="Strategy")
plt.show()

In [ ]:
# =====================================================
# GROWTH CHARTS, ONE PER PERIOD
# =====================================================

fig, axes = plt.subplots(len(PERIODS), 1, figsize=(12, 5 * len(PERIODS)))

for ax, (period_name, results) in zip(axes, all_results.items()):
    for name, result in results.items():
        result.growth.plot(ax=ax, label=name)
    ax.set_title(period_name)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# NOTES
# =====================================================
# Re-read the survivorship bias cell near the top before drawing
# conclusions from the 2010 numbers specifically - they use today's
# ticker list, tested in the past, which flatters the results.
#
# What IS genuinely useful from this comparison: whether a strategy's
# CAGR/Sharpe/drawdown are roughly STABLE as you extend the window, or
# whether they swing wildly. A strategy that only looks good over one
# specific period (say, 2020-present, which includes a huge post-COVID
# recovery rally) and falls apart or reverses over 2010-present is
# telling you something real about how much to trust it.
#
# The data coverage check above is worth actually reading, not
# skipping - if any ticker's data starts well after your requested
# start_date, that ticker was silently excluded from the early part of
# that period's backtest with no error raised.